In [ ]:
1. HR (Heart Rate) — 심박수

단위: BPM (Beats Per Minute)
측정 원리: BVP 신호의 주파수 도메인에서 피크 주파수를 BPM으로 변환
정상 범위: 안정 시 60~100 BPM
임상 의미:
< 60 BPM → 서맥 (Bradycardia)
> 100 BPM → 빈맥 (Tachycardia)



2. SQI (Signal Quality Index) — 신호 품질 지수

범위: 0.0 (노이즈) ~ 1.0 (완벽한 신호)
측정 원리: BVP 신호의 자기상관(Autocorrelation)을 이용해 주기성을 평가
중요성: 다른 모든 지표의 신뢰도를 결정하는 메타 지표
실용적 기준:

≥ 0.7 → 신뢰 가능
0.4~0.7 → 주의 필요
< 0.4 → 해당 윈도우 데이터 사용 지양

⚠️ SQI가 낮을 때의 HRV 값은 생리학적 의미가 없고, 오히려 노이즈입니다.




3. SDNN (Standard Deviation of NN intervals)

단위: ms
측정 원리: RR interval(연속 심박 간격)의 표준편차
정상 범위: 40~100 ms (안정 시 성인)
임상 의미: 전체 자율신경계 활성도를 반영

높을수록 → 건강한 자율신경 균형
낮을수록 → 심부전, 당뇨병성 신경병증, 심근경색 후 위험 상승
< 50 ms → 임상적 주의 기준 (일부 가이드라인)




4. RMSSD (Root Mean Square of Successive Differences)

단위: ms
측정 원리: 연속된 RR interval 차이의 제곱평균제곱근
정상 범위: 20~60 ms
임상 의미: 부교감신경(미주신경) 활성도에 가장 민감한 지표

스트레스/불안 → RMSSD 감소
회복/이완 → RMSSD 증가





5. pNN50

단위: % (퍼센트)
측정 원리: 연속 RR interval 차이가 50ms를 초과하는 쌍의 비율
정상 범위: 10~40%
임상 의미: RMSSD와 유사하게 부교감신경 활성도 반영

고령자, 심혈관 질환자 → 현저히 낮아짐
RMSSD와 높은 상관관계 (사실상 중복 정보가 많음)




6. LF/HF Ratio (Low Frequency / High Frequency)

단위: 무차원 비율
원리: BVP 신호의 주파수 스펙트럼 분석

LF (0.04~0.15 Hz): 교감신경 + 부교감신경 혼합
HF (0.15~0.4 Hz): 부교감신경(호흡과 연동)

정상 범위: 안정 시 1~2, 운동/스트레스 시 급등
임상 의미: 교감/부교감 균형 (자율신경 균형 지표)

높을수록 → 교감신경 우세 (스트레스, 통증, 심부전)
낮을수록 → 부교감 우세 (이완, 수면)
⚠️ 단독으로 사용하면 해석이 모호한 편 (학계에서 논쟁 중)




In [2]:
#pip install open-rppg
import rppg
import time
import cv2

model = rppg.Model('ME-chunk.rlap')

with model.video_capture(0):
    last_process_time = 0
    current_hr = None
    current_sqi = None
    current_hrv = None

    for frame, box in model.preview:
        frame = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)
        h, w = frame.shape[:2]

        # 1초마다 HR + HRV 갱신
        now = time.time()
        if now - last_process_time > 1.0:
            # return_hrv=True가 기본값이므로 생략 가능
            result = model.hr(start=-30)  # HRV 계산은 30초 이상 권장
            if result and result['hr']:
                current_hr  = result['hr']
                current_sqi = result.get('SQI')
                current_hrv = result.get('hrv')
                
                # 콘솔 출력
                print(f"HR: {current_hr:.1f} BPM | SQI: {current_sqi:.2f}")
                if current_hrv:
                    print(
                        #f"  SDNN: {current_hrv.get('sdnn', float('nan')):.1f} ms | "
                        f"RMSSD: {current_hrv.get('rmssd', float('nan')):.1f} ms | "
                        #f"pNN50: {current_hrv.get('pnn50', float('nan')):.1f}% | "
                        f"LF/HF: {current_hrv.get('LF/HF', float('nan')):.2f} | "
                        #f"BR: {current_hrv.get('breathingrate', float('nan')):.1f} /min"
                    )
            last_process_time = now

        # 2. 얼굴 박스 그리기
        if box is not None:
            y1, y2 = box[0]
            x1, x2 = box[1]
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

        # 3. 화면 HUD 표시
        panel_x = 10
        line_h  = 28  # 줄 간격

        def put_label(img, text, row, color=(255, 255, 255)):
            cv2.putText(img, text, (panel_x, 30 + row * line_h),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 0, 0), 4, cv2.LINE_AA)  # 검정 외곽선
            cv2.putText(img, text, (panel_x, 30 + row * line_h),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.65, color,  2, cv2.LINE_AA)

        # HR & SQI
        hr_text  = f"HR : {current_hr:.1f} BPM"  if current_hr  else "HR : --"
        sqi_text = f"SQI: {current_sqi:.2f}"      if current_sqi else "SQI: --"
        sqi_color = (0, 255, 0) if (current_sqi or 0) >= 0.7 else (0, 165, 255)  # 녹색 / 주황

        put_label(frame, hr_text,  row=0, color=(0, 255, 128))
        put_label(frame, sqi_text, row=1, color=sqi_color)

        # HRV 지표
        if current_hrv:
            #sdnn  = current_hrv.get('sdnn',          float('nan'))
            rmssd = current_hrv.get('rmssd',         float('nan'))
            #pnn50 = current_hrv.get('pnn50',         float('nan'))
            lfhf  = current_hrv.get('LF/HF',         float('nan'))
            #br    = current_hrv.get('breathingrate',  float('nan'))

            #put_label(frame, f"SDNN : {sdnn:.1f} ms",   row=3, color=(200, 220, 255))
            put_label(frame, f"RMSSD: {rmssd:.1f} ms",  row=4, color=(200, 220, 255))
            #put_label(frame, f"pNN50: {pnn50:.1f} %",   row=5, color=(200, 220, 255))
            put_label(frame, f"LF/HF: {lfhf:.2f}",      row=6, color=(200, 220, 255))
            #put_label(frame, f"BR   : {br:.1f} /min",   row=7, color=(200, 220, 255))
        else:
            put_label(frame, "HRV : Wait a sec...", row=3, color=(150, 150, 150))

        cv2.imshow("rPPG Monitor", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

HR: 74.5 BPM | SQI: 0.36
HR: 74.5 BPM | SQI: 0.36
HR: 74.5 BPM | SQI: 0.36
HR: 74.5 BPM | SQI: 0.36
HR: 74.5 BPM | SQI: 0.36
HR: 81.8 BPM | SQI: 0.79
  SDNN: 73.8 ms | RMSSD: 39.8 ms | pNN50: 0.2% | LF/HF: 2.68 | BR: 0.3 /min
HR: 81.8 BPM | SQI: 0.79
  SDNN: 73.8 ms | RMSSD: 39.8 ms | pNN50: 0.2% | LF/HF: 2.68 | BR: 0.3 /min
HR: 81.8 BPM | SQI: 0.79
  SDNN: 73.8 ms | RMSSD: 39.8 ms | pNN50: 0.2% | LF/HF: 2.68 | BR: 0.3 /min
HR: 81.8 BPM | SQI: 0.79
  SDNN: 73.8 ms | RMSSD: 39.8 ms | pNN50: 0.2% | LF/HF: 2.68 | BR: 0.3 /min
HR: 81.8 BPM | SQI: 0.79
  SDNN: 73.8 ms | RMSSD: 39.8 ms | pNN50: 0.2% | LF/HF: 2.68 | BR: 0.3 /min
HR: 80.4 BPM | SQI: 0.76
  SDNN: 65.9 ms | RMSSD: 48.3 ms | pNN50: 0.3% | LF/HF: 4.53 | BR: 0.2 /min
HR: 80.4 BPM | SQI: 0.76
  SDNN: 65.9 ms | RMSSD: 48.3 ms | pNN50: 0.3% | LF/HF: 4.53 | BR: 0.2 /min
HR: 80.4 BPM | SQI: 0.76
  SDNN: 65.9 ms | RMSSD: 48.3 ms | pNN50: 0.3% | LF/HF: 4.53 | BR: 0.2 /min
HR: 80.4 BPM | SQI: 0.76
  SDNN: 65.9 ms | RMSSD: 48.3 ms | pNN50: 

OPEN-RPPG:WARNING - Filtering failure.
OPEN-RPPG:WARNING - Filtering failure.
OPEN-RPPG:WARNING - Filtering failure.
OPEN-RPPG:WARNING - Filtering failure.
OPEN-RPPG:WARNING - Filtering failure.


HR: 96.1 BPM | SQI: 0.45
HR: 96.1 BPM | SQI: 0.45
HR: 96.1 BPM | SQI: 0.45
HR: 96.1 BPM | SQI: 0.45
HR: 96.1 BPM | SQI: 0.45
HR: 96.1 BPM | SQI: 0.45
HR: 95.1 BPM | SQI: 0.37
HR: 95.1 BPM | SQI: 0.37
HR: 95.1 BPM | SQI: 0.37
HR: 95.1 BPM | SQI: 0.37
HR: 95.1 BPM | SQI: 0.37
HR: 93.7 BPM | SQI: 0.58
  SDNN: 62.2 ms | RMSSD: 39.9 ms | pNN50: 0.1% | LF/HF: 1.33 | BR: 0.3 /min
HR: 93.7 BPM | SQI: 0.58
  SDNN: 62.2 ms | RMSSD: 39.9 ms | pNN50: 0.1% | LF/HF: 1.33 | BR: 0.3 /min
HR: 93.7 BPM | SQI: 0.58
  SDNN: 62.2 ms | RMSSD: 39.9 ms | pNN50: 0.1% | LF/HF: 1.33 | BR: 0.3 /min
HR: 93.7 BPM | SQI: 0.58
  SDNN: 62.2 ms | RMSSD: 39.9 ms | pNN50: 0.1% | LF/HF: 1.33 | BR: 0.3 /min
HR: 93.7 BPM | SQI: 0.58
  SDNN: 62.2 ms | RMSSD: 39.9 ms | pNN50: 0.1% | LF/HF: 1.33 | BR: 0.3 /min
HR: 94.8 BPM | SQI: 0.64
  SDNN: 90.9 ms | RMSSD: 46.5 ms | pNN50: 0.2% | LF/HF: 2.49 | BR: 0.2 /min
HR: 94.8 BPM | SQI: 0.64
  SDNN: 90.9 ms | RMSSD: 46.5 ms | pNN50: 0.2% | LF/HF: 2.49 | BR: 0.2 /min
HR: 94.8 BPM | SQI

Exception in thread Thread-8:
Traceback (most recent call last):
  File "c:\Users\jisci\miniconda3\envs\ai\lib\threading.py", line 950, in _bootstrap_inner
    self.run()
  File "c:\Users\jisci\miniconda3\envs\ai\lib\site-packages\ipykernel\ipkernel.py", line 772, in run_closure
    _threading_Thread_run(self)
  File "c:\Users\jisci\miniconda3\envs\ai\lib\threading.py", line 888, in run
    self._target(*self._args, **self._kwargs)
  File "c:\Users\jisci\miniconda3\envs\ai\lib\site-packages\rppg\main.py", line 723, in <lambda>
    self.run = threading.Thread(target=lambda:self.__process_video_capture(vid_path, api))
  File "c:\Users\jisci\miniconda3\envs\ai\lib\site-packages\rppg\main.py", line 841, in __process_video_capture
    self.update_frame(img, ts)
  File "c:\Users\jisci\miniconda3\envs\ai\lib\site-packages\rppg\main.py", line 535, in __exit__
    self.wait_completion()
  File "c:\Users\jisci\miniconda3\envs\ai\lib\site-packages\rppg\main.py", line 737, in wait_completion
    s

KeyboardInterrupt: 

In [7]:
import rppg
import time
import cv2
import numpy as np
import pandas as pd
from datetime import datetime

# ── 설정 ───────────────────────────────────────────────
SQI_THRESHOLD   = 0.6    # 이 미만 윈도우 제거
WINDOW_SEC      = 20     # HR/HRV 계산 윈도우 (초)
SAMPLE_INTERVAL = 2      # 샘플링 간격 (초) — 슬라이딩 스텝
BASELINE_N      = 15      # 기준선 추정에 사용할 초기 샘플 수 (= 첫 15초)
OUTPUT_CSV      = f"rppg_clean_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
# ───────────────────────────────────────────────────────

FEATURES = ['hr', 'sqi', 'sdnn', 'rmssd', 'pnn50', 'lf_hf', 'br']

raw_buffer  = []   # SQI 필터링 전 전체 기록
clean_rows  = []   # SQI 통과한 샘플
baseline    = {}   # 개인 기준선 (mean, std)

model = rppg.Model('ME-flow.rlap')

def extract_row(result, ts):
    try:
        hrv = result.get('hrv') or {}
        
        br_raw = hrv.get('breathingrate', float('nan'))
        # Hz로 반환될 경우 /min으로 변환 (0~2 Hz 범위면 Hz로 판단)
        if not np.isnan(br_raw) and br_raw < 2.0:
            br = br_raw * 60
        else:
            br = br_raw

        pnn50_raw = hrv.get('pnn50', float('nan'))
        pnn50 = pnn50_raw * 100 if (not np.isnan(pnn50_raw) and pnn50_raw <= 1.0) else pnn50_raw

        row = {
            'timestamp' : ts,
            'hr'        : result['hr'],
            'sqi'       : result.get('SQI', float('nan')),
            'sdnn'      : hrv.get('sdnn',          float('nan')),
            'rmssd'     : hrv.get('rmssd',         float('nan')),
            'pnn50'     : pnn50,
            'lf_hf'     : hrv.get('LF/HF',         float('nan')),
            'br'        : br,
        }
        if row['hr'] is None or np.isnan(row['hr']):
            return None
        return row
    except (KeyError, TypeError):
        return None


def sqi_filter(row: dict) -> bool:
    """SQI 기준 미달 샘플 제거."""
    return not np.isnan(row['sqi']) and row['sqi'] >= SQI_THRESHOLD


def range_filter(row):
    """생리적으로 불가능한 범위 제거."""
    # 각 피처별 (값, 최솟값, 최댓값) — nan이면 체크 스킵
    checks = [
        (row['hr'],    30,  200),
        (row['sdnn'],   0,  300),
        (row['rmssd'],  0,  300),
        (row['pnn50'],  0,  100),
        (row['lf_hf'],  0,   20),
        (row['br'],     4,   60),   # 이제 /min 단위
    ]
    for val, lo, hi in checks:
        if not np.isnan(val) and not (lo <= val <= hi):
            print(f"[SKIP] 생리 범위 이탈 — "
                  f"HR={row['hr']:.1f}, SDNN={row['sdnn']:.1f}, "
                  f"RMSSD={row['rmssd']:.1f}, LF/HF={row['lf_hf']:.2f}, "
                  f"BR={row['br']:.1f}, PNN50={row['pnn50']:.1f}")
            return False
    return True


def compute_baseline(rows: list[dict]) -> dict:
    """초기 N개 샘플로 기준선(mean, std) 계산."""
    df = pd.DataFrame(rows)[FEATURES]
    return {
        feat: {'mean': df[feat].mean(), 'std': df[feat].std()}
        for feat in FEATURES
    }


def normalize(row: dict, base: dict) -> dict:
    """
    개인 기준선 정규화: z-score = (x - baseline_mean) / baseline_std
    std가 0이면 0으로 처리.
    """
    normed = {'timestamp': row['timestamp']}
    for feat in FEATURES:
        mean = base[feat]['mean']
        std  = base[feat]['std']
        val  = row[feat]
        normed[f'{feat}_raw']    = val
        normed[f'{feat}_znorm']  = (val - mean) / std if std > 0 else 0.0
    return normed


# ── 메인 루프 ───────────────────────────────────────────
print("rPPG 데이터 수집 시작. 'q'로 종료.")

with model.video_capture(0):
    last_sample_time = 0
    baseline_done    = False

    for frame, box in model.preview:
        frame = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)
        now   = time.time()

        # ── 샘플링 ──────────────────────────────────────
        if now - last_sample_time >= SAMPLE_INTERVAL:
            result = model.hr(start=-WINDOW_SEC, return_hrv=True)
            if result:
                row = extract_row(result, ts=now)

                if row:
                    raw_buffer.append(row)  # 원본 보존

                    # ❶ SQI 필터
                    if not sqi_filter(row):
                        print(f"[SKIP] SQI={row['sqi']:.2f} — 신호 품질 미달")
                        last_sample_time = now
                        continue

                    # ❷ 생리 범위 필터
                    if not range_filter(row):
                        print(f"[SKIP] 생리 범위 이탈 — SDNN={row['sdnn']:.1f}, RMSSD={row['rmssd']:.1f}, LF/HF={row['lf_hf']:.2f}, BR={row['br']:.1f}, HR={row['hr']:.1f}, PNN50={row['pnn50']:.1f}")
                        last_sample_time = now
                        continue

                    # ❸ 기준선 수집 단계
                    if not baseline_done:
                        clean_rows.append(row)
                        print(f"[BASELINE {len(clean_rows)}/{BASELINE_N}] HR={row['hr']:.1f}")
                        if len(clean_rows) >= BASELINE_N:
                            baseline = compute_baseline(clean_rows)
                            baseline_done = True
                            print("✅ 기준선 확정:", {k: f"{v['mean']:.2f}±{v['std']:.2f}"
                                                       for k, v in baseline.items()})
                    # ❹ 기준선 이후 → 정규화 & 저장
                    else:
                        normed = normalize(row, baseline)
                        clean_rows.append(normed)
                        print(
                            f"[CLEAN] HR={row['hr']:.1f} | "
                            f"SQI={row['sqi']:.2f} | "
                            f"SDNN={row['sdnn']:.1f} | "
                            f"RMSSD={row['rmssd']:.1f} | "
                            f"LF/HF={row['lf_hf']:.2f} | "
                            f"PNN50={row['pnn50']:.1f} | "
                            f"BR={row['br']:.1f}"
                        )

            last_sample_time = now

        # ── 미니 HUD ────────────────────────────────────
        status = f"Samples: {len(clean_rows)} | Baseline: {'OK' if baseline_done else 'Collecting...'}"
        cv2.putText(frame, status, (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 128), 2)

        if box is not None:
            y1, y2 = box[0]; x1, x2 = box[1]
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

        cv2.imshow("rPPG Collector", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

# ── 저장 ────────────────────────────────────────────────
normalized_rows = [r for r in clean_rows if 'hr_raw' in r]  # 기준선 이후 샘플만

if normalized_rows:
    df_final = pd.DataFrame(normalized_rows)

    # pNN50은 RMSSD와 중복 정보 → 별도 컬럼으로 남기되 주석 표시
    df_final['pnn50_drop_hint'] = True  # EVM 합칠 때 제거 여부 판단용

    df_final.to_csv(OUTPUT_CSV, index=False)
    print(f"\n✅ 저장 완료: {OUTPUT_CSV}")
    print(df_final.describe())
else:
    # raw 원본이라도 저장
    pd.DataFrame(raw_buffer).to_csv("rppg_raw_fallback.csv", index=False)
    print("⚠️  정제 샘플 없음 — raw 데이터만 저장됨")

rPPG 데이터 수집 시작. 'q'로 종료.
[SKIP] SQI=0.47 — 신호 품질 미달
[SKIP] SQI=0.57 — 신호 품질 미달
[SKIP] SQI=0.54 — 신호 품질 미달
[BASELINE 1/15] HR=70.7
[BASELINE 2/15] HR=70.7
[BASELINE 3/15] HR=68.8
[BASELINE 4/15] HR=68.8
[BASELINE 5/15] HR=66.5
[BASELINE 6/15] HR=66.2
[BASELINE 7/15] HR=64.4
[BASELINE 8/15] HR=64.3
[BASELINE 9/15] HR=64.3
[BASELINE 10/15] HR=65.0
[BASELINE 11/15] HR=66.8
[BASELINE 12/15] HR=68.8
[BASELINE 13/15] HR=69.8
[BASELINE 14/15] HR=70.0
[BASELINE 15/15] HR=70.0
✅ 기준선 확정: {'hr': '67.67±2.41', 'sqi': '0.73±0.08', 'sdnn': '74.81±16.25', 'rmssd': '78.32±18.58', 'pnn50': '60.46±13.27', 'lf_hf': '1.73±1.63', 'br': '11.88±5.50'}
[CLEAN] HR=70.0 | SQI=0.79 | SDNN=41.3 | RMSSD=50.3 | LF/HF=0.63 | PNN50=38.1 | BR=19.1
[CLEAN] HR=69.2 | SQI=0.74 | SDNN=41.4 | RMSSD=48.3 | LF/HF=0.41 | PNN50=36.8 | BR=10.0
[CLEAN] HR=69.1 | SQI=0.71 | SDNN=80.2 | RMSSD=48.2 | LF/HF=0.34 | PNN50=35.3 | BR=7.1
[CLEAN] HR=68.8 | SQI=0.66 | SDNN=100.3 | RMSSD=45.9 | LF/HF=0.68 | PNN50=33.3 | BR=11.5
[SKIP] SQI=0

Exception in thread Thread-23:
Traceback (most recent call last):
  File "c:\Users\jisci\miniconda3\envs\ai\lib\threading.py", line 950, in _bootstrap_inner
    self.run()
  File "c:\Users\jisci\miniconda3\envs\ai\lib\site-packages\ipykernel\ipkernel.py", line 772, in run_closure
    _threading_Thread_run(self)
  File "c:\Users\jisci\miniconda3\envs\ai\lib\threading.py", line 888, in run
    self._target(*self._args, **self._kwargs)
  File "c:\Users\jisci\miniconda3\envs\ai\lib\site-packages\rppg\main.py", line 723, in <lambda>
    self.run = threading.Thread(target=lambda:self.__process_video_capture(vid_path, api))
  File "c:\Users\jisci\miniconda3\envs\ai\lib\site-packages\rppg\main.py", line 841, in __process_video_capture
    self.update_frame(img, ts)
  File "c:\Users\jisci\miniconda3\envs\ai\lib\site-packages\rppg\main.py", line 535, in __exit__
    self.wait_completion()
  File "c:\Users\jisci\miniconda3\envs\ai\lib\site-packages\rppg\main.py", line 737, in wait_completion
    

KeyboardInterrupt: 